# Project 5: Neural Network from Scratch

**Chapters:** 6 & 7 — Neural Networks, SGD & Backprop

**Goal:** Build a 2-layer neural network (1 hidden layer) from scratch. Implement forward pass, activation functions, backpropagation, and SGD. Solve the XOR problem that stumped the perceptron.

**What you'll build:**
1. Forward pass with ReLU and sigmoid activations
2. Backpropagation (manual gradient computation)
3. SGD with mini-batches
4. Nonlinear decision boundaries

**Rules:** No sklearn, no PyTorch/TF. NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. Data

In [ ]:
def make_moons(n=300, noise=0.15, seed=42):
    rng = np.random.default_rng(seed)
    n_half = n // 2
    theta = np.linspace(0, np.pi, n_half)
    X_top = np.column_stack([np.cos(theta), np.sin(theta)]) + rng.normal(scale=noise, size=(n_half, 2))
    X_bot = np.column_stack([1 - np.cos(theta), 1 - np.sin(theta) - 0.5]) + rng.normal(scale=noise, size=(n_half, 2))
    X = np.vstack([X_top, X_bot])
    y = np.array([0]*n_half + [1]*n_half)  # using 0/1 for cross-entropy
    idx = rng.permutation(n)
    return X[idx], y[idx]

def make_spiral(n=300, seed=42):
    rng = np.random.default_rng(seed)
    n_half = n // 2
    theta = np.linspace(0, 2*np.pi, n_half) + rng.normal(scale=0.15, size=n_half)
    r = np.linspace(0.5, 2, n_half)
    X_a = np.column_stack([r*np.cos(theta), r*np.sin(theta)])
    X_b = np.column_stack([r*np.cos(theta + np.pi), r*np.sin(theta + np.pi)])
    X = np.vstack([X_a, X_b])
    y = np.array([0]*n_half + [1]*n_half)
    idx = rng.permutation(n)
    return X[idx], y[idx]

def plot_data(X, y, ax=None, title=""):
    if ax is None: fig, ax = plt.subplots()
    ax.scatter(X[y==0,0], X[y==0,1], c='steelblue', marker='o', label='0', edgecolors='k', linewidth=0.5, s=30, alpha=0.7)
    ax.scatter(X[y==1,0], X[y==1,1], c='salmon', marker='s', label='1', edgecolors='k', linewidth=0.5, s=30, alpha=0.7)
    ax.set_title(title); ax.legend(); ax.set_aspect('equal')
    return ax

X_train, y_train = make_moons(n=300)
plot_data(X_train, y_train, title="Moons Dataset")
plt.show()

## 2. Build the Network — Your Turn

### Architecture

```
Input (2) → Hidden (hidden_size, ReLU) → Output (1, Sigmoid) → Binary prediction
```

### The math

**Forward pass:**
- $\mathbf{z}_1 = X \mathbf{W}_1 + \mathbf{b}_1$
- $\mathbf{a}_1 = \text{ReLU}(\mathbf{z}_1)$
- $z_2 = \mathbf{a}_1 \mathbf{w}_2 + b_2$
- $\hat{y} = \sigma(z_2)$

**Loss:** Binary cross-entropy
$$L = -\frac{1}{n}\sum [y \log(\hat{y}) + (1-y)\log(1-\hat{y})]$$

**Backprop:** Chain rule through each layer.

In [ ]:
def relu(z):
    """ReLU activation: max(0, z)"""
    ### YOUR CODE ###
    raise NotImplementedError

def relu_derivative(z):
    """Derivative of ReLU: 1 if z > 0, else 0"""
    ### YOUR CODE ###
    raise NotImplementedError

def sigmoid(z):
    """Sigmoid: 1 / (1 + exp(-z)). Clip z to avoid overflow."""
    ### YOUR CODE ###
    raise NotImplementedError

In [ ]:
class TwoLayerNet:
    def __init__(self, n_input=2, n_hidden=16, lr=0.1):
        rng = np.random.default_rng(42)
        # He initialization for ReLU
        self.W1 = rng.normal(scale=np.sqrt(2/n_input), size=(n_input, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.w2 = rng.normal(scale=np.sqrt(2/n_hidden), size=(n_hidden,))
        self.b2 = 0.0
        self.lr = lr
        self.loss_history = []
    
    def forward(self, X):
        """Forward pass. Store intermediates for backprop.
        
        Returns:
            y_hat: predicted probabilities, shape (n,)
        """
        ### YOUR CODE ###
        # self.z1 = X @ W1 + b1
        # self.a1 = relu(z1)
        # self.z2 = a1 @ w2 + b2
        # y_hat = sigmoid(z2)
        raise NotImplementedError("Forward pass")
    
    def compute_loss(self, y_hat, y):
        """Binary cross-entropy loss."""
        ### YOUR CODE ###
        # Clip y_hat to avoid log(0)
        raise NotImplementedError("BCE loss")
    
    def backward(self, X, y, y_hat):
        """Backpropagation. Compute gradients and update weights.
        
        Chain rule:
        dL/dz2 = y_hat - y                           (n,)
        dL/dw2 = a1.T @ dz2 / n                      (hidden,)
        dL/db2 = mean(dz2)
        dL/da1 = dz2 (outer) w2                       (n, hidden)
        dL/dz1 = dL/da1 * relu'(z1)                   (n, hidden)
        dL/dW1 = X.T @ dz1 / n                        (input, hidden)
        dL/db1 = mean(dz1, axis=0)
        """
        ### YOUR CODE ###
        raise NotImplementedError("Backprop")
    
    def train_step(self, X, y):
        """One forward + backward pass."""
        y_hat = self.forward(X)
        loss = self.compute_loss(y_hat, y)
        self.backward(X, y, y_hat)
        self.loss_history.append(loss)
        return loss
    
    def predict(self, X):
        probs = self.forward(X)
        return (probs >= 0.5).astype(int)
    
    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

### Sanity checks

In [ ]:
# Activation tests
assert np.allclose(relu(np.array([-1, 0, 1])), [0, 0, 1])
assert np.allclose(relu_derivative(np.array([-1, 0, 1])), [0, 0, 1])
assert abs(sigmoid(0) - 0.5) < 1e-6
print("✓ Activations correct")

# Forward pass should produce values in (0, 1)
net_test = TwoLayerNet(n_input=2, n_hidden=4, lr=0.1)
probs = net_test.forward(X_train[:5])
assert probs.shape == (5,), f"Output shape should be (5,), got {probs.shape}"
assert all(0 < p < 1 for p in probs), "Probabilities must be in (0, 1)"
print("✓ Forward pass correct")

# Training should reduce loss
loss_before = net_test.compute_loss(net_test.forward(X_train), y_train)
for _ in range(50):
    net_test.train_step(X_train, y_train)
loss_after = net_test.loss_history[-1]
assert loss_after < loss_before, f"Loss should decrease: {loss_before:.4f} → {loss_after:.4f}"
print(f"✓ Loss decreased: {loss_before:.4f} → {loss_after:.4f}")

print("\n🎉 All checks passed!")

### Gradient checking

The most important debugging tool for backprop: compare your analytical gradients with a numerical approximation.

$$\frac{\partial L}{\partial \theta} \approx \frac{L(\theta + h) - L(\theta - h)}{2h}$$

If the relative difference is $> 10^{-5}$, your backward pass has a bug.

In [ ]:
def numerical_gradient(net, X, y, param_name, idx, h=1e-5):
    """Compute numerical gradient for one parameter element."""
    param = getattr(net, param_name)
    orig = param[idx] if np.ndim(param) > 0 else param

    # f(theta + h)
    if np.ndim(param) > 0:
        param[idx] = orig + h
    else:
        setattr(net, param_name, orig + h)
    y_hat_plus = net.forward(X)
    loss_plus = net.compute_loss(y_hat_plus, y)

    # f(theta - h)
    if np.ndim(param) > 0:
        param[idx] = orig - h
    else:
        setattr(net, param_name, orig - h)
    y_hat_minus = net.forward(X)
    loss_minus = net.compute_loss(y_hat_minus, y)

    # Restore
    if np.ndim(param) > 0:
        param[idx] = orig
    else:
        setattr(net, param_name, orig)

    return (loss_plus - loss_minus) / (2 * h)


# Run gradient check on a small network
gc_net = TwoLayerNet(n_input=2, n_hidden=4, lr=0.1)
X_gc, y_gc = X_train[:5], y_train[:5]

# Forward + backward to compute analytical gradients
y_hat_gc = gc_net.forward(X_gc)
loss_gc = gc_net.compute_loss(y_hat_gc, y_gc)
n_gc = len(y_gc)

# Compute analytical gradients (without updating weights)
dz2 = y_hat_gc - y_gc
dw2_analytical = gc_net.a1.T @ dz2 / n_gc
db2_analytical = np.mean(dz2)
da1 = np.outer(dz2, gc_net.w2)
dz1 = da1 * relu_derivative(gc_net.z1)
dW1_analytical = X_gc.T @ dz1 / n_gc
db1_analytical = np.mean(dz1, axis=0)

# Check a few parameters from each weight matrix
print("Gradient check (analytical vs numerical):")
print(f"{'Param':<10} {'Analytical':>12} {'Numerical':>12} {'Rel Diff':>12}")
print("-" * 50)

checks_passed = True
for param_name, analytical in [('W1', dW1_analytical), ('w2', dw2_analytical),
                                ('b1', db1_analytical), ('b2', db2_analytical)]:
    if np.ndim(analytical) == 0:
        num = numerical_gradient(gc_net, X_gc, y_gc, param_name, None)
        ana = float(analytical)
        rel_diff = abs(ana - num) / max(abs(ana), abs(num), 1e-8)
        status = "OK" if rel_diff < 1e-5 else "FAIL"
        print(f"{param_name:<10} {ana:>12.8f} {num:>12.8f} {rel_diff:>12.2e} {status}")
        if rel_diff >= 1e-5: checks_passed = False
    else:
        idxs = [(0,0), (1,0)] if np.ndim(analytical) == 2 else [(0,), (1,)]
        for idx in idxs:
            num = numerical_gradient(gc_net, X_gc, y_gc, param_name, idx)
            ana = analytical[idx]
            rel_diff = abs(ana - num) / max(abs(ana), abs(num), 1e-8)
            status = "OK" if rel_diff < 1e-5 else "FAIL"
            print(f"{param_name}{list(idx)!s:<6} {ana:>12.8f} {num:>12.8f} {rel_diff:>12.2e} {status}")
            if rel_diff >= 1e-5: checks_passed = False

assert checks_passed, "Gradient check failed — your backward pass has a bug!"
print("\n✓ All gradient checks passed!")

## 3. Train & Visualize

In [ ]:
net = TwoLayerNet(n_input=2, n_hidden=32, lr=0.5)

for epoch in range(1, 1001):
    loss = net.train_step(X_train, y_train)
    if epoch <= 5 or epoch % 200 == 0:
        print(f"Epoch {epoch:4d}  |  loss: {loss:.4f}  |  acc: {net.accuracy(X_train, y_train):.1%}")

In [ ]:
def plot_nn_boundary(model, X, y, title="", ax=None, resolution=150):
    if ax is None: fig, ax = plt.subplots()
    xmin, xmax = X[:,0].min()-0.5, X[:,0].max()+0.5
    ymin, ymax = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, resolution),
                         np.linspace(ymin, ymax, resolution))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=['#cce5ff', '#ffcccc'], alpha=0.4)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    plot_data(X, y, ax=ax, title=title)
    return ax

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_nn_boundary(net, X_train, y_train, title=f"Moons (acc={net.accuracy(X_train, y_train):.0%})", ax=ax1)
ax2.plot(net.loss_history)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.set_title('Training Loss'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Experiments

### A: Hidden layer size
Try n_hidden = 2, 4, 16, 64. When does it solve moons? When does it start overfitting?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
for h, ax in zip([2, 4, 16, 64], axes):
    m = TwoLayerNet(n_input=2, n_hidden=h, lr=0.5)
    for _ in range(1000): m.train_step(X_train, y_train)
    plot_nn_boundary(m, X_train, y_train, title=f"hidden={h}, acc={m.accuracy(X_train, y_train):.0%}", ax=ax)
plt.tight_layout(); plt.show()

**Your observation:** *(How does hidden size affect the boundary complexity?)*

...

### Debugging challenge
The network below has THREE bugs in its backward pass. Can you spot them without running the code? Then run it and see what happens. (This is what real ML debugging looks like.)

*Hints: think about signs, derivatives, and normalization.*

In [ ]:
class BuggyNet:
    """A neural net with 3 bugs in backward(). Find and describe them below."""
    def __init__(self, n_input=2, n_hidden=16, lr=0.1):
        rng = np.random.default_rng(99)
        self.W1 = rng.normal(scale=np.sqrt(2/n_input), size=(n_input, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.w2 = rng.normal(scale=np.sqrt(2/n_hidden), size=(n_hidden,))
        self.b2 = 0.0
        self.lr = lr
    
    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)  # ReLU
        self.z2 = self.a1 @ self.w2 + self.b2
        y_hat = 1 / (1 + np.exp(-np.clip(self.z2, -500, 500)))
        return y_hat
    
    def backward(self, X, y, y_hat):
        n = X.shape[0]
        # Bug 1: wrong sign in output gradient
        dz2 = y - y_hat                            # BUG: should be (y_hat - y)
        
        dw2 = self.a1.T @ dz2 / n
        db2 = np.mean(dz2)
        
        # Bug 2: missing ReLU derivative
        da1 = np.outer(dz2, self.w2)
        dz1 = da1                                   # BUG: missing * (self.z1 > 0)
        
        # Bug 3: wrong matrix multiplication order
        dW1 = dz1.T @ X / n                         # BUG: should be X.T @ dz1 / n
        db1 = np.mean(dz1, axis=0)
        
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2

# Try training the buggy net — observe what goes wrong
buggy = BuggyNet(n_input=2, n_hidden=16, lr=0.1)
losses = []
for ep in range(200):
    yh = buggy.forward(X_train)
    eps_clip = 1e-7
    loss = -np.mean(y_train * np.log(yh + eps_clip) + (1 - y_train) * np.log(1 - yh + eps_clip))
    buggy.backward(X_train, y_train, yh)
    losses.append(loss)

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Buggy Network — Loss should decrease but...')
plt.grid(True, alpha=0.3); plt.show()

# YOUR ANSWERS:
# Bug 1: ___
# Bug 2: ___
# Bug 3: ___

### B: Try the spiral dataset
This is *much* harder. Can your 2-layer net solve it?

In [ ]:
X_sp, y_sp = make_spiral(n=400)
net_sp = TwoLayerNet(n_input=2, n_hidden=64, lr=0.5)
for epoch in range(1, 3001):
    net_sp.train_step(X_sp, y_sp)
    if epoch % 500 == 0:
        print(f"Epoch {epoch}  |  loss: {net_sp.loss_history[-1]:.4f}  |  acc: {net_sp.accuracy(X_sp, y_sp):.1%}")

plot_nn_boundary(net_sp, X_sp, y_sp, title=f"Spiral (acc={net_sp.accuracy(X_sp, y_sp):.0%})")
plt.show()

### C: Why normalization matters
What happens when features have very different scales? Gradient descent struggles because the loss surface becomes elongated — large features dominate the gradient.

In [ ]:
# Create data with badly scaled features
X_raw, y_raw = make_moons(n=300)
X_bad = X_raw.copy()
X_bad[:, 0] *= 100  # feature 0 is 100x larger than feature 1

# Standardize: zero mean, unit variance
X_norm = X_bad.copy()
X_norm = (X_norm - X_norm.mean(axis=0)) / X_norm.std(axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Train on raw (badly scaled) data
net_raw = TwoLayerNet(n_input=2, n_hidden=32, lr=0.001)  # small lr to avoid divergence
for _ in range(500): net_raw.train_step(X_bad, y_raw)
ax1.plot(net_raw.loss_history, label=f'Unnormalized (final={net_raw.loss_history[-1]:.3f})')

# Train on normalized data
net_norm = TwoLayerNet(n_input=2, n_hidden=32, lr=0.5)
for _ in range(500): net_norm.train_step(X_norm, y_raw)
ax1.plot(net_norm.loss_history, label=f'Normalized (final={net_norm.loss_history[-1]:.3f})')

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Effect of Normalization')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.bar(['Unnormalized', 'Normalized'],
        [net_raw.accuracy(X_bad, y_raw), net_norm.accuracy(X_norm, y_raw)],
        color=['salmon', 'steelblue'])
ax2.set_ylabel('Accuracy'); ax2.set_title('Final Accuracy (500 epochs)')
ax2.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()

### D: Learning curves
How much data does the network need? Plot train and test accuracy as a function of training set size. This is the most diagnostic plot in ML — it tells you whether you need more data, more capacity, or more regularization.

In [ ]:
# Generate a large dataset, split into train/test
X_lc, y_lc = make_moons(n=400, seed=42)
X_lc_test, y_lc_test = make_moons(n=200, seed=99)

sizes = [20, 50, 100, 200, 300, 400]
train_accs, test_accs = [], []

for n in sizes:
    net_lc = TwoLayerNet(n_input=2, n_hidden=32, lr=0.5)
    for _ in range(500):
        net_lc.train_step(X_lc[:n], y_lc[:n])
    train_accs.append(net_lc.accuracy(X_lc[:n], y_lc[:n]))
    test_accs.append(net_lc.accuracy(X_lc_test, y_lc_test))

plt.figure(figsize=(8, 5))
plt.plot(sizes, train_accs, 'bo-', label='Train accuracy')
plt.plot(sizes, test_accs, 'rs-', label='Test accuracy')
plt.xlabel('Training set size'); plt.ylabel('Accuracy')
plt.title('Learning Curves — Neural Network on Moons')
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0.5, 1.05)
plt.show()

## 5. What You Learned

1. What does the hidden layer "learn" that a linear model can't?
2. Why is weight initialization important? (Try all-zeros and see.)
3. What's the relationship between hidden size and decision boundary complexity?
4. Why is the spiral harder than the moons?

---
**Next project:** RNN — apply neural networks to sequences.